# Condições atmosféricas de um lançamento pela reanálise ERA5

Este notebook obtém a partir da reanálise ERA5 as condições atmosféricas de um lançamento já realizado, no formato lido pelo RocketPy. Ele segue o procedimento descrito no apêndice "Procedimento para caracterização de condições atmosféricas pela reanálise ERA5" do trabalho de conclusão de curso da autora:

1. **Estimativa:** A partir da posição e da altitude do trilho, do horário e do apogeu, estima as pressões no trilho e no apogeu pela atmosfera padrão (ISO 2533), com um fator de segurança sobre o apogeu. Com isso, escolhe os níveis de pressão e monta a requisição ao Climate Data Store (CDS).
2. **Conferência:** Lê o arquivo NetCDF baixado e calcula a altitude real de cada nível na posição do trilho, em cada horário. Confirma que há pelo menos um nível abaixo do trilho e que o nível superior fica acima do apogeu com o fator de segurança. Se uma das condições falhar, indica o nível a acrescentar e envia a requisição corrigida ao CDS.
3. **Relevo do modelo:** Baixa o geopotencial da superfície do ERA5 na mesma área e verifica se os níveis usados na interpolação até o trilho estão acima do relevo do modelo, onde os valores não são extrapolados.

A estimativa usa a atmosfera padrão e pode deixar de fora parte do voo. Quem garante que os níveis baixados cobrem todo o voo, do trilho ao apogeu com o fator de segurança, é a conferência, feita com os dados do dia do lançamento. Com a sua chave do CDS, o notebook baixa o arquivo, confere e, se faltar algum nível, faz uma nova requisição, sem necessidade de intervenção.

In [ ]:
!pip install -q xarray netCDF4 cdsapi

## Entradas

In [ ]:
ENTRADAS = dict(
    nome        = "Juno II",
    lat         = -23.36346389,   # latitude do trilho (graus, positiva ao norte)
    lon         = -48.01255833,   # longitude do trilho (graus, positiva a leste)
    h_t         = 665.0,          # altitude do trilho em relação ao nível do mar (metros)
    data_utc    = "2022-08-05",   # data do lançamento em UTC (AAAA-MM-DD)
    hora_utc    = "20:26",        # instante do lançamento em UTC (HH:MM)
    dh_apo      = 3706.0,         # altura do apogeu acima do trilho (metros)
    origem_apo  = "barometro",    # "barometro" (altímetro barométrico), "gnss" ou "previsto" (simulação, voo sem dados)
    margem_area = 0.5,            # folga mínima da área em torno do trilho (graus; pelo menos 0,25)
)

## Pasta de trabalho

Os arquivos baixados são salvos na pasta indicada em `PASTA_TRABALHO`. Para salvá-los numa pasta do Google Drive:

1. Escreva em `PASTA_TRABALHO` o caminho da pasta a partir de **Meu Drive**. Se ela ainda não existir, será criada.
2. Rode esta célula e autorize o acesso ao Drive quando o Colab pedir.
3. Confira, na saída da célula, o caminho completo da pasta de trabalho.

Se `PASTA_TRABALHO` ficar vazia, os arquivos vão para `/content` e são apagados quando a sessão do Colab termina. Fora do Colab, `PASTA_TRABALHO` é um caminho no próprio computador. Se estiver vazia, os arquivos ficam na pasta do notebook.

In [ ]:
PASTA_TRABALHO = ""   # ex.: "Juno_II/atmosfera_era5"

import os

if "PASTA_INICIAL" not in globals():    # pasta em que a sessão começou
    PASTA_INICIAL = os.getcwd()

try:
    import google.colab
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB and PASTA_TRABALHO:
    if not os.path.isdir("/content/drive/MyDrive"):   # monta o Drive só na primeira vez
        from google.colab import drive
        drive.mount("/content/drive")
    if PASTA_TRABALHO.startswith("/content/drive"):   # caminho completo também é aceito
        pasta = PASTA_TRABALHO
    else:
        pasta = os.path.join("/content/drive/MyDrive", PASTA_TRABALHO.strip("/"))
    os.makedirs(pasta, exist_ok=True)
    os.chdir(pasta)
elif NO_COLAB:
    os.chdir("/content")
    print("Atenção: PASTA_TRABALHO vazia. Os arquivos ficarão em /content e serão "
          "apagados quando a sessão do Colab terminar.")
elif PASTA_TRABALHO:
    pasta = os.path.join(PASTA_INICIAL, PASTA_TRABALHO)
    os.makedirs(pasta, exist_ok=True)
    os.chdir(pasta)
else:
    os.chdir(PASTA_INICIAL)

print("Pasta de trabalho:", os.getcwd())

## Atmosfera padrão

Esta célula não precisa ser alterada. Ela define a atmosfera padrão (ISO 2533), um modelo de como a pressão diminui com a altitude, usado para estimar quais níveis de pressão do ERA5 cobrem o voo. Os parâmetros são os mesmos usados pelo RocketPy.

A estimativa é só um ponto de partida. No dia do lançamento, a pressão e a temperatura reais diferem das do modelo, e os níveis ficam um pouco mais altos ou mais baixos que o previsto. Por isso, depois do download, o notebook confere os níveis com os dados reais e, se faltar algum nível, faz uma nova requisição automaticamente.

In [ ]:
import numpy as np, math, datetime as dt

G_N, R_AR, P_N = 9.80665, 287.05287, 1013.25          # m/s², J/(kg K), hPa
H_B  = [0.0, 11e3, 20e3, 32e3, 47e3]                  # m geopotencial
T_B  = [288.15, 216.65, 216.65, 228.65, 270.65]       # K
BETA = [-6.5e-3, 0.0, 1.0e-3, 2.8e-3, 0.0]            # K/m

def _camada(p_b, T_b, beta, dH):
    if beta != 0:
        return p_b * (1 + beta / T_b * dH) ** (-G_N / (beta * R_AR))
    return p_b * math.exp(-G_N * dH / (R_AR * T_b))

P_B = [P_N]
for i in range(len(H_B) - 1):
    P_B.append(_camada(P_B[i], T_B[i], BETA[i], H_B[i + 1] - H_B[i]))

def raio_terra(lat_graus):
    # Raio do elipsoide na latitude dada (m), como no RocketPy 1.13.0.
    a, b = 6378137.0, 6356752.314245
    f = math.radians(lat_graus)
    num = (a**2 * math.cos(f))**2 + (b**2 * math.sin(f))**2
    den = (a * math.cos(f))**2 + (b * math.sin(f))**2
    return math.sqrt(num / den)

def geometrica_para_geopotencial(h, R_T):
    return R_T * h / (R_T + h)

def geopotencial_para_geometrica(H, R_T):
    return R_T * H / (R_T - H)

def p_padrao(h, R_T):
    # Pressão (hPa) na altitude geométrica h (m), pela atmosfera padrão.
    H = geometrica_para_geopotencial(h, R_T)
    if not 0 <= H < 51e3:
        raise ValueError("Altitude fora do intervalo de 0 a 51 km da atmosfera padrão.")
    i = max(k for k in range(len(H_B)) if H_B[k] <= H)
    return _camada(P_B[i], T_B[i], BETA[i], H - H_B[i])

## 1. Estimativa e escolha dos níveis

A estimativa usa a atmosfera padrão para escolher uma faixa contínua de níveis de pressão do ERA5 que cubra o voo, de um extremo ao outro:

- **Nível inferior:** o de menor pressão entre os que superam a pressão estimada no trilho, ou seja, o nível logo abaixo do trilho;
- **Nível superior:** o de maior pressão entre os que ficam abaixo da pressão estimada no apogeu com o fator de segurança, ou seja, o nível logo acima do apogeu com o fator de segurança.

A altura do apogeu é incerta, e por isso é multiplicada por um fator de segurança $F$, escolhido pela entrada `origem_apo`. O resultado é o apogeu com o fator de segurança, $h_t + F\,\Delta h_{\text{apogeu}}$:

- **`"barometro"`**: $F = 1{,}1$. Altímetros barométricos calculam a altura a partir da pressão, supondo a atmosfera padrão. Quando a temperatura do dia difere da padrão, a altura medida erra cerca de 4 % a cada 10 °C de diferença (EUROCONTROL, 2014, com base no ICAO Doc 8168). O fator 1,1 cobre, assim, diferenças de até cerca de 25 °C;
- **`"gnss"`**: $F = 1$, porque a medição não depende da pressão;
- **`"previsto"`** (apogeu só simulado, voo sem dados): $F = 2$, que cobre um voo de até o dobro da altura prevista. Cada nível a mais acrescenta só alguns quilobytes ao arquivo e não altera o perfil usado no RocketPy.

Como a pressão e a temperatura do dia deslocam a altitude dos níveis, a escolha pode falhar. Nesse caso, a conferência da Seção 2 acusa a falha, e o notebook acrescenta um nível no lado que falhou e refaz a requisição.

In [ ]:
NIVEIS_ERA5 = [1000, 975, 950, 925, 900, 875, 850, 825, 800, 775, 750, 700, 650,
               600, 550, 500, 450, 400, 350, 300, 250, 225, 200, 175, 150, 125,
               100, 70, 50, 30, 20, 10, 7, 5, 3, 2, 1]   # hPa

F_APOGEU = {"barometro": 1.1, "gnss": 1.0, "previsto": 2.0}

E = ENTRADAS
if E["origem_apo"] not in F_APOGEU:
    raise ValueError(f'origem_apo deve ser um destes valores: {", ".join(F_APOGEU)}.')
R_T = raio_terra(E["lat"])
F = F_APOGEU[E["origem_apo"]]
h_apo_real = E["h_t"] + E["dh_apo"]   # altitude do apogeu informado
h_apo = E["h_t"] + F * E["dh_apo"]    # altitude do apogeu com o fator de segurança
p_inf = p_padrao(E["h_t"], R_T)       # pressão no trilho, atmosfera padrão
p_sup = p_padrao(h_apo, R_T)          # pressão no apogeu com o fator de segurança

def escolher_niveis(p_trilho, p_apogeu):
    acima = [n for n in NIVEIS_ERA5 if n > p_trilho]
    abaixo = [n for n in NIVEIS_ERA5 if n < p_apogeu]
    if not acima:
        raise ValueError("Nenhum nível do ERA5 tem pressão maior que a do trilho.")
    if not abaixo:
        raise ValueError("O apogeu com o fator de segurança fica acima do nível de 1 hPa, o mais alto do ERA5.")
    inf, sup = min(acima), max(abaixo)
    return [n for n in NIVEIS_ERA5 if sup <= n <= inf]

niveis = escolher_niveis(p_inf, p_sup)

print(f"{E['nome']} (altitudes acima do nível do mar):")
print(f"  Trilho:                           {E['h_t']:6.0f} m   (pressão estimada: {p_inf:.1f} hPa)")
print(f"  Apogeu informado:                 {h_apo_real:6.0f} m   ({E['origem_apo']}, F = {F:g})")
print(f"  Apogeu com o fator de segurança:  {h_apo:6.0f} m   (pressão estimada: {p_sup:.1f} hPa)")
print(f"  Níveis: {niveis[0]} a {niveis[-1]} hPa ({len(niveis)} níveis)")

### Horários e área

O ERA5 fornece dados de hora em hora, numa grade de 0,25° (cerca de 25 km). Esta célula define o recorte de tempo e de espaço a ser baixado.

- **Horários:** como o lançamento raramente coincide com um horário cheio, entram as duas horas que o cercam, a anterior e a posterior à `hora_utc`, o que permite usar a mais próxima ou interpolar entre elas. Se o lançamento for entre 23h e 0h UTC, elas caem em dias diferentes, e ambos os dias são incluídos.
- **Área:** um retângulo com limites nos pontos da grade e folga de pelo menos `margem_area` em cada direção (0,5° por padrão, ajustável na célula **Entradas**). Como o cálculo no trilho usa apenas o ponto vizinho de cada lado, a folga padrão deixa um ponto a mais, o que permite corrigir essa posição em até 0,25° sem baixar os dados de novo.

In [ ]:
t_L = dt.datetime.fromisoformat(f"{E['data_utc']}T{E['hora_utc']}")
t_antes = t_L.replace(minute=0, second=0, microsecond=0)
t_depois = t_antes + dt.timedelta(hours=1)
if t_L == t_antes:
    horarios = [t_antes]
else:
    horarios = [t_antes, t_depois]

def multiplo(x, sentido):
    q = x / 0.25
    return (math.ceil(q - 1e-9) if sentido > 0 else math.floor(q + 1e-9)) * 0.25

m = E["margem_area"]
if m < 0.25:
    raise ValueError("margem_area deve ser de pelo menos 0,25° (um espaçamento da grade).")
area = dict(norte=multiplo(E["lat"] + m, +1), sul=multiplo(E["lat"] - m, -1),
            oeste=multiplo(E["lon"] - m, -1), leste=multiplo(E["lon"] + m, +1))
folga = dict(norte=area["norte"] - E["lat"], sul=E["lat"] - area["sul"],
             oeste=E["lon"] - area["oeste"], leste=area["leste"] - E["lon"])

print(f"{E['nome']}:")
print(f"  Lançamento:  {t_L:%d/%m/%Y %H:%M} UTC")
print(f"  Horários:    " + ", ".join(f"{h:%d/%m/%Y %Hh}" for h in horarios) + " UTC")
print(f"  Área (graus, folga em relação ao trilho):")
for lado in ("norte", "sul", "oeste", "leste"):
    print(f"    {lado.capitalize() + ':':7s} {area[lado]:7.2f}   (folga {folga[lado]:.3f})")

## 2. Download e conferência

**Acesso ao CDS:**
1. Crie uma conta no [Climate Data Store](https://cds.climate.copernicus.eu), se ainda não tiver. Depois, abra as páginas [*ERA5 hourly data on pressure levels*](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-pressure-levels) e [*ERA5 hourly data on single levels*](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels) e aceite a licença de cada uma na aba *Download*, no fim do formulário. Isso só precisa ser feito uma vez.
2. Copie a sua chave (*Personal Access Token*) na página [How to use the CDS API](https://cds.climate.copernicus.eu/how-to-api), com a sua conta conectada. Ela aparece no quadro de configuração, na linha `key:`.
3. Guarde a chave nos *Secrets* do Colab:
   1. Na barra lateral esquerda, clique no ícone de chave (**Secrets**).
   2. Clique em **Adicionar novo secret** (*Add new secret*).
   3. Em **Nome** (*Name*), escreva `CDSAPI_KEY`, exatamente assim.
   4. Em **Valor** (*Value*), cole a chave copiada no passo 2.
   5. Ative a chave **Acesso ao notebook** (*Notebook access*) ao lado do segredo.

Assim, a chave não fica escrita no notebook e não vai junto se ele for compartilhado. O segredo fica salvo na sua conta do Google e vale para os próximos usos.

**Modos de execução:**

Com `USAR_API = True` (definido na célula abaixo), o notebook faz tudo sozinho:

1. Faz a requisição ao CDS com os níveis de pressão estimados e baixa o arquivo.
2. Confere, em cada horário, se há pelo menos um nível abaixo do trilho e se o nível superior fica acima do apogeu com o fator de segurança.
3. Se alguma condição falhar, acrescenta o nível seguinte do ERA5 no extremo que falhou e refaz a requisição.
4. Repete até as duas condições serem cumpridas, no máximo `MAX_TENTATIVAS` vezes.

Com `USAR_API = False`, só a parte de criação de conta é necessária. O notebook mostra a requisição para ser feita no site e depois solicita o envio do arquivo baixado. Se a conferência falhar, ele mostra a requisição corrigida. Nesse caso, faça a nova requisição no site e rode a célula de novo.

In [ ]:
import textwrap, unicodedata

USAR_API        = True
MAX_TENTATIVAS  = 4

DATASET = "reanalysis-era5-pressure-levels"
VARIAVEIS = ["geopotential", "temperature", "u_component_of_wind",
             "v_component_of_wind", "specific_humidity", "relative_humidity",
             "fraction_of_cloud_cover", "specific_cloud_liquid_water_content",
             "specific_cloud_ice_water_content", "specific_rain_water_content",
             "specific_snow_water_content"]
BASE = "era5_" + unicodedata.normalize("NFKD", E["nome"]).encode("ascii", "ignore").decode().lower().replace(" ", "_")

def montar_requisicao(niveis):
    dias = sorted({h.date() for h in horarios})
    return {
        "product_type": ["reanalysis"],
        "variable": VARIAVEIS,
        "year": sorted({f"{d.year}" for d in dias}),
        "month": sorted({f"{d.month:02d}" for d in dias}),
        "day": sorted({f"{d.day:02d}" for d in dias}),
        "time": sorted({h.strftime("%H:00") for h in horarios}),
        "pressure_level": [str(n) for n in niveis],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [area["norte"], area["oeste"], area["sul"], area["leste"]],
    }

def imprimir_requisicao(req):
    # Mostra a requisição campo a campo, com listas longas quebradas em várias linhas.
    for campo, valor in req.items():
        texto = ", ".join(map(str, valor)) if isinstance(valor, list) else str(valor)
        linhas = textwrap.wrap(texto, width=60) or [""]
        print(f"  {campo:16s} {linhas[0]}")
        for linha in linhas[1:]:
            print(" " * 19 + linha)

def cliente_cds():
    import cdsapi
    from google.colab import userdata
    try:
        chave = userdata.get("CDSAPI_KEY")
    except Exception:
        raise RuntimeError("Chave do CDS não encontrada. Crie o segredo CDSAPI_KEY e "
                           "libere o acesso deste notebook (veja 'Acesso ao CDS').") from None
    return cdsapi.Client(url="https://cds.climate.copernicus.eu/api", key=chave)

def baixar(niveis):
    arquivo = f"{BASE}_pressoes.nc"
    cliente_cds().retrieve(DATASET, montar_requisicao(niveis), arquivo)
    return arquivo

In [ ]:
import xarray as xr

def conferir(arquivo, mostrar=True):
    # Altitude de cada nível na posição do trilho, em cada horário de interesse.
    ds = xr.open_dataset(arquivo)
    dim_t = "valid_time" if "valid_time" in ds.dims else "time"
    dim_p = "pressure_level" if "pressure_level" in ds.dims else "level"
    try:
        ds = ds.sel({dim_t: [np.datetime64(h) for h in horarios]})
    except KeyError:
        raise ValueError("O arquivo não contém os horários "
                         + ", ".join(f"{h:%d/%m/%Y %Hh}" for h in horarios) + " UTC.") from None
    lats, lons = ds["latitude"].values, ds["longitude"].values
    if not (lats.min() < E["lat"] < lats.max() and lons.min() < E["lon"] < lons.max()):
        raise ValueError("O trilho está fora da área do arquivo ou na borda dela.")
    p = ds[dim_p].values.astype(float)
    ordem = np.argsort(-p)
    p = p[ordem]
    horas = []
    for t in ds[dim_t].values:
        z = ds["z"].sel({dim_t: t})
        if "number" in z.dims:
            z = z.isel(number=0)
        z = z.interp(latitude=E["lat"], longitude=E["lon"], method="linear").values[ordem]
        h = geopotencial_para_geometrica(z / G_N, R_T)
        r = dict(t=t, h=h, ok_inf=bool((h < E["h_t"]).any()), ok_sup=bool(h[-1] > h_apo))
        horas.append(r)
        if mostrar:
            rotulo = str(np.datetime_as_string(t, unit="h")).replace("T", " ") + "h UTC"
            abaixo = ", ".join(f"{n:.0f}" for n in p[h < E["h_t"]]) or "nenhum"
            print(f"  {rotulo}:")
            print(f"    Níveis abaixo do trilho ({E['h_t']:.0f} m): {abaixo} hPa"
                  f"  {'OK' if r['ok_inf'] else 'FALHOU'}")
            print(f"    Nível superior: {p[-1]:.0f} hPa, a {h[-1]:.0f} m "
                  f"(apogeu com o fator de segurança a {h_apo:.0f} m)  {'OK' if r['ok_sup'] else 'FALHOU'}")
    ds.close()
    return p, horas

def ampliar(niveis, horas):
    # Acrescenta o nível seguinte do ERA5 em cada extremo que falhou.
    novos = list(niveis)
    if not all(r["ok_inf"] for r in horas):
        i = NIVEIS_ERA5.index(novos[0])
        if i == 0:
            raise RuntimeError("O trilho fica abaixo do nível de 1000 hPa, o mais baixo do ERA5.")
        novos.insert(0, NIVEIS_ERA5[i - 1])
        print(f"  acrescentando {NIVEIS_ERA5[i - 1]} hPa abaixo do trilho")
    if not all(r["ok_sup"] for r in horas):
        i = NIVEIS_ERA5.index(novos[-1])
        if i == len(NIVEIS_ERA5) - 1:
            raise RuntimeError("O apogeu com o fator de segurança fica acima do nível de 1 hPa, o mais alto do ERA5.")
        novos.append(NIVEIS_ERA5[i + 1])
        print(f"  acrescentando {NIVEIS_ERA5[i + 1]} hPa acima do apogeu com o fator de segurança")
    return novos

In [ ]:
ARQUIVO = None

if USAR_API:
    niveis_atuais = list(niveis)
    for tentativa in range(1, MAX_TENTATIVAS + 1):
        print(f"Tentativa {tentativa}: níveis {niveis_atuais[0]} a {niveis_atuais[-1]} hPa")
        arquivo = baixar(niveis_atuais)
        print("-" * 60 + "\n Conferência:")
        p_arq, horas = conferir(arquivo)
        if all(r["ok_inf"] and r["ok_sup"] for r in horas):
            ARQUIVO = arquivo
            print(f"\nNíveis adequados. Arquivo final: {ARQUIVO}")
            break
        niveis_atuais = ampliar(niveis_atuais, horas)
    else:
        print("\nAs condições não foram cumpridas no número máximo de tentativas.")
else:
    from google.colab import files
    print("Requisição com os níveis estimados (se esta célula já indicou uma requisição "
          "corrigida, envie o arquivo dela):")
    imprimir_requisicao(montar_requisicao(niveis))
    enviado = next(iter(files.upload()))
    p_arq, horas = conferir(enviado)
    if all(r["ok_inf"] and r["ok_sup"] for r in horas):
        ARQUIVO = f"{BASE}_pressoes.nc"
        os.replace(enviado, ARQUIVO)
        print(f"\nNíveis adequados. Arquivo final: {ARQUIVO}")
    else:
        niveis_arquivo = [int(n) for n in p_arq]
        niveis_corrigidos = ampliar(niveis_arquivo, horas)
        print("\nRefaça a requisição com estes níveis e rode esta célula de novo:")
        imprimir_requisicao(montar_requisicao(niveis_corrigidos))

## 3. Relevo do modelo

O ERA5 tem um relevo próprio, uma espécie de média do terreno dentro de cada célula da grade, que pode ficar acima ou abaixo da altitude real do trilho. Quando um nível de pressão fica abaixo desse relevo, o modelo não tem dados nesse ponto, e o ECMWF preenche os valores por extrapolação ao converter os resultados para níveis de pressão. Como as condições no trilho são interpoladas entre o nível logo abaixo e o logo acima dele, um valor preenchido nesses níveis afeta o próprio trilho.

Esta célula obtém o relevo do modelo no conjunto *ERA5 hourly data on single levels*, na mesma área, e verifica, em cada horário e nos quatro pontos da grade em torno do trilho, se esses dois níveis estão acima dele:

- **Nenhum valor extrapolado:** as condições no trilho vêm inteiramente do modelo;
- **Há valores extrapolados:** as condições no trilho dependem em parte do preenchimento, e convém considerar essa limitação ao usá-las.

Esta célula usa a licença do conjunto *single levels*, aceita na configuração de **Acesso ao CDS** (Seção 2).

In [ ]:
CONFERIR_SUPERFICIE = True
DATASET_SUP = "reanalysis-era5-single-levels"

def requisicao_superficie():
    # O geopotencial da superfície não varia no tempo; basta um horário.
    t = horarios[0]
    return {
        "product_type": ["reanalysis"],
        "variable": ["geopotential"],
        "year": [f"{t.year}"],
        "month": [f"{t.month:02d}"],
        "day": [f"{t.day:02d}"],
        "time": [t.strftime("%H:00")],
        "data_format": "netcdf",
        "download_format": "unarchived",
        "area": [area["norte"], area["oeste"], area["sul"], area["leste"]],
    }

def vizinhos(valores, x):
    # Coordenadas da grade imediatamente abaixo e acima de x.
    return sorted({float(valores[valores <= x].max()), float(valores[valores >= x].min())})

SUPERFICIE = None
if ARQUIVO and CONFERIR_SUPERFICIE:
    arq_sup = f"{BASE}_relevo.nc"
    if USAR_API:
        cliente_cds().retrieve(DATASET_SUP, requisicao_superficie(), arq_sup)
        print("-" * 60)
    else:
        from google.colab import files
        print("Faça esta requisição no conjunto ERA5 hourly data on single levels e envie o arquivo baixado:")
        imprimir_requisicao(requisicao_superficie())
        enviado = next(iter(files.upload()))
        os.replace(enviado, arq_sup)

    # Altitude do relevo do modelo nos quatro pontos vizinhos e no trilho
    sup = xr.open_dataset(arq_sup)
    zs = sup["z"]
    for d in ("valid_time", "time", "number"):
        if d in zs.dims:
            zs = zs.isel({d: 0})
    v_lat = vizinhos(zs["latitude"].values, E["lat"])
    v_lon = vizinhos(zs["longitude"].values, E["lon"])
    h_sup_viz = geopotencial_para_geometrica(
        zs.sel(latitude=v_lat, longitude=v_lon).values / G_N, R_T)
    h_sup = float(geopotencial_para_geometrica(
        zs.interp(latitude=E["lat"], longitude=E["lon"]).values / G_N, R_T))
    sup.close()

    print("Relevo do modelo (altitudes acima do nível do mar):")
    print(f"  Na posição do trilho:  {h_sup:.0f} m (trilho a {E['h_t']:.0f} m; diferença {E['h_t'] - h_sup:+.0f} m)")
    print("  Nos quatro pontos vizinhos: " + ", ".join(f"{x:.0f}" for x in h_sup_viz.ravel()) + " m")

    # Níveis usados na interpolação até o trilho, em cada horário de interesse
    ds = xr.open_dataset(ARQUIVO)
    dim_t = "valid_time" if "valid_time" in ds.dims else "time"
    dim_p = "pressure_level" if "pressure_level" in ds.dims else "level"
    ds = ds.sel({dim_t: [np.datetime64(h) for h in horarios]})
    p = ds[dim_p].values.astype(float)
    ordem = np.argsort(-p)
    p = p[ordem]
    SUPERFICIE = dict(h_sup=h_sup, h_sup_viz=h_sup_viz, horas=[])
    for t in ds[dim_t].values:
        z = ds["z"].sel({dim_t: t})
        if "number" in z.dims:
            z = z.isel(number=0)
        h_viz = geopotencial_para_geometrica(
            z.sel(latitude=v_lat, longitude=v_lon).values[ordem] / G_N, R_T)
        h_tr = geopotencial_para_geometrica(
            z.interp(latitude=E["lat"], longitude=E["lon"]).values[ordem] / G_N, R_T)
        k_inf = int(np.where(h_tr < E["h_t"])[0].max())
        usados = []
        for k in (k_inf, k_inf + 1):
            n = int((h_viz[k] < h_sup_viz).sum())
            usados.append(dict(p=p[k], h=h_tr[k], n_abaixo=n, n_total=h_sup_viz.size))
        todos_abaixo = [int(p[k]) for k in range(len(p)) if (h_viz[k] < h_sup_viz).any()]
        SUPERFICIE["horas"].append(dict(t=t, usados=usados, niveis_abaixo=todos_abaixo))
        rotulo = str(np.datetime_as_string(t, unit="h")).replace("T", " ") + "h UTC"
        print(f"\n  {rotulo}:")
        for u, nome in zip(usados, ("logo abaixo do trilho", "logo acima do trilho")):
            print(f"    {u['p']:.0f} hPa, {nome} (a {u['h']:.0f} m): "
                  f"abaixo do relevo em {u['n_abaixo']} de {u['n_total']} pontos")
        lista = ", ".join(str(n) for n in todos_abaixo) + " hPa" if todos_abaixo else "nenhum"
        print(f"    Níveis abaixo do relevo em algum ponto: {lista}")
    ds.close()

    if all(u["n_abaixo"] == 0 for r in SUPERFICIE["horas"] for u in r["usados"]):
        print("\nNenhum valor extrapolado entra na interpolação até o trilho.")
    else:
        print("\nAtenção: a interpolação até o trilho usa valores extrapolados abaixo do relevo do modelo.")

## 4. Resumo e arquivo final

A célula mostra os valores da verificação no horário mais próximo do lançamento e onde o arquivo final foi salvo. Com `BAIXAR_NO_COMPUTADOR = True`, o arquivo também é baixado pelo navegador.

In [ ]:
BAIXAR_NO_COMPUTADOR = False   # True: baixa também o arquivo final pelo navegador

if ARQUIVO:
    p_arq, horas = conferir(ARQUIVO, mostrar=False)
    ref = min(horas, key=lambda r: abs(r["t"] - np.datetime64(t_L)))
    h = ref["h"]
    pz = lambda alt: float(np.interp(alt, h, p_arq))
    k = int((h < E["h_t"]).sum())
    rotulo = str(np.datetime_as_string(ref["t"], unit="h")).replace("T", " ") + "h UTC"

    print(f"{E['nome']}, {rotulo} (altitudes acima do nível do mar):")
    print(f"  Níveis no arquivo:               {p_arq[0]:.0f} a {p_arq[-1]:.0f} hPa ({len(p_arq)} níveis)")
    print(f"  Nível inferior:                  {p_arq[0]:.0f} hPa, a {h[0]:.0f} m")
    print(f"  Primeiro nível acima do trilho:  {p_arq[k]:.0f} hPa, a {h[k]:.0f} m")
    print(f"  Nível superior:                  {p_arq[-1]:.0f} hPa, a {h[-1]:.0f} m")
    print(f"  Pressão no trilho:               {pz(E['h_t']):.1f} hPa")
    print(f"  Pressão no apogeu informado:     {pz(h_apo_real):.1f} hPa")
    if globals().get("SUPERFICIE") is not None:
        extrapolado = any(u["n_abaixo"] for r in SUPERFICIE["horas"] for u in r["usados"])
        print(f"  Relevo do modelo:                "
              + ("Há valores extrapolados até o trilho" if extrapolado else "Nenhum valor extrapolado até o trilho"))

    print("\nArquivo salvo em:", os.path.abspath(ARQUIVO))

    if BAIXAR_NO_COMPUTADOR and NO_COLAB:
        from google.colab import files
        files.download(ARQUIVO)
else:
    print("Nenhum arquivo passou na conferência; veja as mensagens da Seção 2.")